In [1]:
import json
import os
import kagglehub
import importlib
import torch
import random
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re
import pandas as pd
import numpy as np
import gc
from src.utils import load_indices_from_jsonl
from IPython.display import JSON
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from functools import partial
from src.jsonl_dataset import JsonLDataset
from transformers import AutoTokenizer, TrainingArguments, EvalPrediction
from adapters import AutoAdapterModel, AdapterTrainer
from sklearn.metrics import f1_score, precision_score, recall_score
from adapters.composition import Parallel 

### Setup

In [7]:
path = kagglehub.dataset_download("Cornell-University/arxiv/versions/272")
ds_path = os.path.join(path, 'arxiv-metadata-oai-snapshot.json')
master_ds = JsonLDataset(ds_path)
augmented_ds = JsonLDataset("resources/augmented_index.jsonl")

Indexing dataset at C:\Users\kerem\.cache\kagglehub\datasets\Cornell-University\arxiv\versions\272\arxiv-metadata-oai-snapshot.json... (this may take a minute)
Indexed 2951540 entries.
Indexing dataset at resources/augmented_index.jsonl... (this may take a minute)
Indexed 2951540 entries.


In [3]:
train_indices = load_indices_from_jsonl("resources/train_indicies_parent_categories.jsonl", flatten=True, shuffle=True, seed=42)
test_indices = load_indices_from_jsonl("resources/test_indices_parent_categories.jsonl", flatten=True, shuffle=True, seed=42)

In [2]:
from src.multiadapter_parent_predictor import MultiAdapterParentPredictor

In [4]:
from sklearn.preprocessing import MultiLabelBinarizer

# 1. Define your Fixed Taxonomy
TARGET_PARENT_CLASSES = [
    'Physics', 
    'Mathematics', 
    'Computer Science', 
    'Quantitative Biology', 
    'Statistics', 
    'Quantitative Finance', 
    'Economics', 
    'Electrical Engineering and Systems Science'
]

def create_mlb(target_classes):
    mlb = MultiLabelBinarizer(classes=target_classes)
    mlb.fit([target_classes])
    return mlb

mlb_parent = create_mlb(TARGET_PARENT_CLASSES)

In [6]:
def get_x_y_for_pipeline(pipeline):
    BATCH_SIZE = 16 
    
    master_samples = [master_ds[idx] for idx in test_indices]
    ground_truth_labels = [augmented_ds[idx][2] for idx in test_indices]
    
    x = []
    y = mlb_parent.transform(ground_truth_labels)
    
    for i in tqdm(range(0, len(master_samples), BATCH_SIZE)):
        batch = master_samples[i : i + BATCH_SIZE]
        preds = pipeline.predict(batch)
        
        for p in preds:
            x.append(p['labels_mlb'])

    return x, y

### Parent pipeline eval with 1 adapter

In [5]:
parent_inference_pipeline = MultiAdapterParentPredictor(
    adapter_configs=[
    {"path": "./resources/parent_categories_adapter_bucket2/", "name": "arxiv_parent_categories_classifier_bucket2"}
], mlb=mlb_parent)

Initializing Multi-Adapter Pipeline on cuda...
  -> Loading arxiv_parent_categories_classifier_bucket2...


There are adapters available but none are activated for the forward pass.


In [38]:
x, y = get_x_y_for_pipeline(parent_inference_pipeline)

100%|██████████████████████████████████████████████████████████████████████████████████████████| 994/994 [08:14<00:00,  2.01it/s]


In [39]:
assert np.squeeze(np.array(x), axis=1).shape == np.array(y).shape
print(f"Evaluation of the inference pipeline with 1 adapters")
print(f"F1: {f1_score(np.squeeze(np.array(x), axis=1), np.array(y), average='micro')}")
print(f"Precision: {precision_score(np.squeeze(np.array(x), axis=1), np.array(y), average='micro')}")
print(f"Recall: {recall_score(np.squeeze(np.array(x), axis=1), np.array(y), average='micro')}")

Evaluation of the inference pipeline with 1 adapters
F1: 0.8324559267100062
Precision: 0.8075972373682297
Recall: 0.8588935709591371


In [16]:
gc.collect()
torch.cuda.empty_cache()

### Parent pipeline eval with 2 adapters

In [38]:
parent_inference_pipeline2 = MultiAdapterParentPredictor(adapter_configs=[
    {"path": "./resources/parent_categories_adapter/", "name": "arxiv_parent_categories_classifier"},
    {"path": "./resources/parent_categories_adapter_bucket2/", "name": "arxiv_parent_categories_classifier_bucket2"}
], mlb=mlb_parent)

Initializing Multi-Adapter Pipeline on cuda...


There are adapters available but none are activated for the forward pass.


  -> Loading arxiv_parent_categories_classifier...
  -> Loading arxiv_parent_categories_classifier_bucket2...


In [172]:
x, y = get_x_y_for_pipeline(parent_inference_pipeline2)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\preprocessing\_label.py:900: UserWarning: unknown class(es) ['Other/Legacy'] will be ignored
  warnings.warn(
100%|██████████████████████████████████████████████████████████████████████████████████████████| 994/994 [11:08<00:00,  1.49it/s]


In [203]:
assert np.squeeze(np.array(x), axis=1).shape == np.array(y).shape
print(f"Evaluation of the inference pipeline with 2 adapters")
print(f"F1: {f1_score(np.squeeze(np.array(x), axis=1), np.array(y), average='micro')}")
print(f"Precision: {precision_score(np.squeeze(np.array(x), axis=1), np.array(y), average='micro')}")
print(f"Recall: {recall_score(np.squeeze(np.array(x), axis=1), np.array(y), average='micro')}")

Evaluation of the inference pipeline with 2 adapters
F1: 0.8398896304858496
Precision: 0.8187931661214104
Recall: 0.8621019595835885


### Parent pipeline eval with 3 adapters

In [7]:
parent_inference_pipeline3 = MultiAdapterParentPredictor(adapter_configs=[
    {"path": "./resources/parent_categories_adapter/", "name": "arxiv_parent_categories_classifier"},
    {"path": "./resources/parent_categories_adapter_bucket2/", "name": "arxiv_parent_categories_classifier_bucket2"},
    {"path": "./resources/parent_categories_adapter_bucket3/", "name": "arxiv_parent_categories_classifier_bucket3"}
], mlb=mlb_parent)

Initializing Multi-Adapter Pipeline on cuda...
  -> Loading arxiv_parent_categories_classifier...
  -> Loading arxiv_parent_categories_classifier_bucket2...


There are adapters available but none are activated for the forward pass.


  -> Loading arxiv_parent_categories_classifier_bucket3...


In [8]:
x, y = get_x_y_for_pipeline(parent_inference_pipeline3)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\preprocessing\_label.py:900: UserWarning: unknown class(es) ['Other/Legacy'] will be ignored
  warnings.warn(
100%|████████████████████████████████████████████████████████████████████████████████| 994/994 [21:09<00:00,  1.28s/it]


In [10]:
assert np.squeeze(np.array(x), axis=1).shape == np.array(y).shape
print(f"Evaluation of the inference pipeline with 3 adapters")
print(f"F1: {f1_score(np.squeeze(np.array(x), axis=1), np.array(y), average='micro')}")
print(f"Precision: {precision_score(np.squeeze(np.array(x), axis=1), np.array(y), average='micro')}")
print(f"Recall: {recall_score(np.squeeze(np.array(x), axis=1), np.array(y), average='micro')}")

Evaluation of the inference pipeline with 3 adapters
F1: 0.8395250433388634
Precision: 0.8185750636132315
Recall: 0.8615755442476183


### Test output format

In [42]:
parent_inference_pipeline.predict([master_ds[2]])

[{'logits': [array([[9.9531364e-01, 3.1082283e-03, 3.9515821e-03, 4.4666114e-03,
           2.3204819e-03, 8.4731332e-04, 4.2179614e-04, 9.3358371e-04]],
         dtype=float32)],
  'probabilities': [0.9953136444091797,
   0.003108228323981166,
   0.003951582126319408,
   0.004466611426323652,
   0.0023204819299280643,
   0.0008473133202642202,
   0.0004217961395625025,
   0.0009335837094113231],
  'labels': ['Physics'],
  'labels_mlb': array([[1, 0, 0, 0, 0, 0, 0, 0]])}]

In [40]:
parent_inference_pipeline2.predict([master_ds[2]])

[{'logits': [array([[9.9829477e-01, 2.6484155e-03, 1.1952696e-03, 1.3247031e-03,
           1.0303751e-03, 2.2938354e-04, 1.1201695e-04, 6.2273961e-04]],
         dtype=float32),
   array([[9.9531353e-01, 3.1082283e-03, 3.9515803e-03, 4.4666175e-03,
           2.3204831e-03, 8.4731425e-04, 4.2179614e-04, 9.3358371e-04]],
         dtype=float32)],
  'probabilities': [0.9968041181564331,
   0.002878321800380945,
   0.0025734249502420425,
   0.0028956602327525616,
   0.001675429055467248,
   0.0005383489187806845,
   0.00026690654340200126,
   0.0007781616877764463],
  'labels': ['Physics'],
  'labels_mlb': array([[1, 0, 0, 0, 0, 0, 0, 0]])}]

In [41]:
parent_inference_pipeline3.predict([master_ds[2]])

[{'logits': [array([[9.9829477e-01, 2.6484155e-03, 1.1952696e-03, 1.3247031e-03,
           1.0303751e-03, 2.2938354e-04, 1.1201695e-04, 6.2273961e-04]],
         dtype=float32),
   array([[9.9531353e-01, 3.1082283e-03, 3.9515803e-03, 4.4666175e-03,
           2.3204831e-03, 8.4731425e-04, 4.2179614e-04, 9.3358371e-04]],
         dtype=float32),
   array([[9.9335432e-01, 3.0987202e-03, 2.6973751e-03, 8.8199964e-03,
           1.7113880e-03, 7.2039547e-04, 6.0184568e-04, 7.4726593e-04]],
         dtype=float32)],
  'probabilities': [0.9956542253494263,
   0.0029517882503569126,
   0.00261474191211164,
   0.004870438948273659,
   0.0016874154098331928,
   0.0005990311037749052,
   0.00037855294067412615,
   0.0007678631227463484],
  'labels': ['Physics'],
  'labels_mlb': array([[1, 0, 0, 0, 0, 0, 0, 0]])}]

### Manual evaluation of the predictive quality

In [165]:
rand_idx = random.choice(test_indices)
rand_sample = master_ds[rand_idx]
res = parent_inference_pipeline2.predict([rand_sample])
print(f"Predicted: {res[0]['labels']}")
print(f"Actual   : {augmented_ds[rand_idx][2]}")
print(f"Score:   : {f1_score(res[0]['labels_mlb'], mlb_parent.transform([augmented_ds[rand_idx][2]]), average="micro")}")

Predicted: ['Physics']
Actual   : ['Physics']
Score:   : 1.0


In [144]:
res[0]['labels_mlb']

array([[0, 0, 0, 1, 0, 0, 0, 0]])

In [146]:
mlb_parent.transform([augmented_ds[rand_idx][2]])

array([[1, 0, 0, 1, 0, 0, 0, 0]])

In [149]:
f1_score([[1, 0, 0, 1, 0, 0, 0, 0]], [[1, 0, 0, 1, 0, 0, 0, 0]], average="micro")

1.0

In [120]:
res[0]['labels_mlb']

array([[0, 0, 1, 0, 0, 1, 0, 0]])

In [112]:
mlb_parent.transform([res[0]['labels']])

array([[0, 0, 0, 0, 1, 0, 0, 0]])

In [111]:
mlb_parent.transform([augmented_ds[rand_idx][2]])

array([[0, 0, 0, 0, 1, 0, 0, 0]])

### Manual evaluation of logits aggregation

In [60]:
res[0]['logits'][0][0].tolist()

[0.02983766235411167,
 0.004058793652802706,
 0.9026868343353271,
 0.7659959197044373,
 0.029498063027858734,
 0.0003198585473001003,
 0.0004093350435141474,
 0.7059654593467712]

In [61]:
res[0]['logits'][1][0].tolist()

[0.02192239835858345,
 0.0069285291247069836,
 0.9079641103744507,
 0.6884737610816956,
 0.041807886213064194,
 0.0008135340758599341,
 0.00047387508675456047,
 0.7104717493057251]

In [65]:
((res[0]['logits'][0][0] + res[0]['logits'][1][0]) / 2).tolist()

[0.025880031287670135,
 0.005493661388754845,
 0.9053254723548889,
 0.7272348403930664,
 0.035652972757816315,
 0.0005666962824761868,
 0.0004416050505824387,
 0.7082185745239258]

In [66]:
res[0]['probabilities']

[0.025880031287670135,
 0.005493661388754845,
 0.9053254723548889,
 0.7272348403930664,
 0.035652972757816315,
 0.0005666962824761868,
 0.0004416050505824387,
 0.7082185745239258]